**Provision Resources On Chameleon (run in Chameleon Juypeter Environment)**

In [ ]:
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")

In [ ]:
l = lease.Lease(f"lease-icp-{username}", duration=datetime.timedelta(hours=48))
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.xlarge"), amount=1)
l.submit(idempotent=True)

In [ ]:
l.show()

In [ ]:
s = server.Server(
    f"node-icp-{username}",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.show(type="widget")

In [ ]:
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-8888", "port": 8888, "description": "Enable TCP port 8888 (Jupyter)"},
]

In [ ]:
for sg in security_groups:
    secgroup = network.SecurityGroup(
        {
            "name": sg["name"],
            "description": sg["description"],
        }
    )
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")

In [ ]:
s.refresh()
s.check_connectivity()

**Clone repo**

In [ ]:
s.execute("git clone https://github.com/ashleyogorman/ICP_Milestone_2.git")

**Set up Docker**

In [ ]:
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")

**Open SSH**

In [ ]:
ip = s.get_floating_ip()
print(f"ssh -i ~/.ssh/id_rsa_chameleon cc@{ip}")

Run:

    ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D

in local terminal with floating IP from above

**If ssh error**

In [ ]:
print(f"ssh-keygen -R {ip}")

**Build project in SSH Session**

    cd ICP_Milestone_1
    docker build -t icp-pipeline .
    docker run -p 8888:8888 icp-pipeline

Copy and paste

    http://127.0.0.1:8888/tree?token=...

into URL, swapping 127.0.0.1 with your own floating IP

**Build & Run With a Volume Mount — For Development Only**

    cd ~/ICP_Milestone_1
    docker build -t icp-pipeline .
    docker run -p 8888:8888 -v ~/ICP_Milestone_1:/icp-milestone-1 icp-pipeline

Saving progress on session

    git add .
    git commit -m "your message here"
    git push

Note:
- You will need a personal access token to push in this manner
- If making edits to this file, do not push sensitive information to this notebook or repo